# Label masks as plain TIFF or OME-TIFF -> precomputed raster + meshes

The melanoma and invasive notebooks read the axis order from the OME-XML of an OME-TIFF. A plain `.tiff` has no OME-XML (`TiffFile.ome_metadata` is `None`), so that code fails. This notebook uses `convert_mask_to_precomputed_raster_and_meshes()`, which detects the kind of mask and converts it accordingly:

- **OME-TIFF**: the axis order and voxel sizes are read from the file.
- **plain TIFF**: the stack axis is assumed to be `z` (with a warning); pass `dims=` and `voxel_size_um=` if that is not right.
- **OME-Zarr**: converted directly.

The detection uses the file content, so an OME-TIFF named `*.tif` is still recognized.

## 0. Setup

In [ ]:
from pathlib import Path

from tissue_map_tools.masks import detect_mask_format, read_tiff_mask
from tissue_map_tools.igneous_converters import (
    from_mask_to_precomputed_raster_and_meshes,
)
from tissue_map_tools.view import view_precomputed_in_neuroglancer

dataset_path = Path.cwd().parent / "data" / "plain_tiff"
print(dataset_path)
out_path = dataset_path / "out"
out_path.mkdir(parents=True, exist_ok=True)
raw_path = mask_path = dataset_path / "raw"

mask_path = raw_path / "mask.tiff"
precomputed_path = out_path / "mask_precomputed"

## 1. Download and prepare the plain mask tiff file

In [ ]:
import tifffile

download_path = str(dataset_path / "z-series.ome.tiff")
# 1. download (or just save the file from the link in your browser)
import urllib.request
url = "https://downloads.openmicroscopy.org/images/OME-TIFF/2016-06/bioformats-artificial/z-series.ome.tiff"
urllib.request.urlretrieve(url, download_path)


arr = tifffile.imread(download_path)
print(arr.dtype, arr.min(), arr.max())   # int8, -1 ... ?

# test mask: threshold into labels 0/1
mask = (arr > arr.mean()).astype("uint16")
tifffile.imwrite(mask_path, mask)


## 2. Inspect the mask

`detect_mask_format()` returns `"ome-tiff"`, `"tiff"` or `"ome-zarr"`. For a TIFF, `read_tiff_mask()` shows the axes and voxel sizes that will be used, before converting anything.

In [ ]:
print("format:", detect_mask_format(mask_path))

mask = read_tiff_mask(mask_path)  # lazy, nothing is loaded in memory
print("dims:", mask.dims)
print("shape:", mask.data.shape, "chunks:", mask.data.chunksize)
print("voxel size (um):", mask.voxel_size_um or "not stored in the file")

## 3. Convert

For a plain TIFF the axis order and the voxel size are not stored in the file. If the stack axis is not `z`, or you know the physical resolution, pass them explicitly, for example:

```python
convert_mask_to_precomputed_raster_and_meshes(
    mask_path, precomputed_path,
    dims=("z", "y", "x"),
    voxel_size_um={"x": 0.5, "y": 0.5, "z": 2.0},
)
```


In [ ]:
if not (precomputed_path / "info").exists():
    from_mask_to_precomputed_raster_and_meshes(
        mask_path=mask_path,
        precomputed_path=precomputed_path,
        intermediate_zarr_path=out_path / "mask.zarr",
        shape=(128, 128, 128),
        nlod=3,
        min_chunk_size=(32, 32, 32),
        parallel=1,
    )
    print("Conversion complete.")
else:
    print("Precomputed output already exists -- skipping conversion.")

## 4. Visualize in Vitessce

In [ ]:
import vitessce

# Stop previously started local servers before re-running
vitessce.data_server.stop_all()

from tissue_map_tools.utils import compute_initial_camera_state
from tissue_map_tools.vitessce_configs.layer_specs import SegmentationLayerSpec
from tissue_map_tools.vitessce_configs.neuroglancer_config_builder import (
    build_vitessce_config,
)

use_web_app = True
initial_camera_state = compute_initial_camera_state(data_path=str(precomputed_path))

vc = build_vitessce_config(
    name="Precomputed data",
    segmentations=[
        SegmentationLayerSpec(
            file_uid="segmentation",
            data_path=str(precomputed_path),
        ),
    ],
    initial_camera_state=initial_camera_state,
    use_web_app=use_web_app,
)

vc

## 5. Visualize in Neuroglancer

In [ ]:
viewer = view_precomputed_in_neuroglancer(data_path=str(precomputed_path))
viewer